# CadQuery: Parametric CAD

## Overview
CadQuery is a Python library for building parametric 3D CAD models. It provides:
- Sketch-based modeling (lines, arcs, splines)
- Extrusion, revolution, lofting
- Boolean operations (union, difference, intersection)
- Export to STEP, STL, DXF

## Why CadQuery for Engineering?
- Create parametric geometry for CFD simulations
- Automate geometry generation
- Integrate with meshing tools (gmsh)

In [ ]:
import cadquery as cq
import matplotlib.pyplot as plt
import numpy as np

# 1. Basic Sketching
print("=== Basic Sketching ===")

# Create a simple rectangle
result = cq.Workplane("XY").rect(2, 1).extrude(0.5)

print(f"Created a 2x1x0.5 box")
print(f"Volume: {result.val().Volume():.3f}")

# Create a circle
result_circle = cq.Workplane("XY").circle(1).extrude(0.5)
print(f"\nCreated a cylinder (r=1, h=0.5)")
print(f"Volume: {result_circle.val().Volume():.3f}")

# 2. Complex Sketches
print("\n=== Complex Sketches ===")

# Create an airfoil-like shape
def airfoil_spline(t):
    """Simple airfoil upper surface"""
    x = t
    y = 0.15 * np.sqrt(t) * (1 - t)  # Simple camber line
    return (x, y)

# Create a parametric airfoil
n_points = 20
t_vals = np.linspace(0, 1, n_points)
upper_points = [airfoil_spline(t) for t in t_vals]
lower_points = [(x, -0.05 * np.sqrt(t) * (1 - t)) for t, (x, _) in zip(t_vals, upper_points)]

# Combine points (closed loop: upper surface + reversed lower surface)
# Remove duplicate start/end point
airfoil_points = upper_points + lower_points[::-1][:-1]

result_airfoil = (cq.Workplane("XY")
                  .spline(airfoil_points, periodic=True)
                  .extrude(1.0))

print(f"Created a parametric airfoil")
print(f"Volume: {result_airfoil.val().Volume():.3f}")

## Boolean Operations

Combine shapes using union, difference, and intersection.

In [ ]:
# 3. Boolean Operations
print("\n=== Boolean Operations ===")

# Create a box with a hole
box = cq.Workplane("XY").box(2, 2, 1)
hole = cq.Workplane("XY").circle(0.5).extrude(1.0)

result_hole = box.cut(hole)
print(f"Box with hole")
print(f"Volume: {result_hole.val().Volume():.3f}")

# Create a union of shapes
sphere = cq.Workplane("XY").sphere(0.5)
result_union = box.union(sphere)
print(f"\nBox + Sphere (union)")
print(f"Volume: {result_union.val().Volume():.3f}")

# 4. Export to STL
print("\n=== Export to STL ===")

# Export the airfoil to STL
cq.exporters.export(result_airfoil, "airfoil.stl")
print(f"Exported airfoil to airfoil.stl")

# Export to STEP
cq.exporters.export(result_airfoil, "airfoil.step")
print(f"Exported airfoil to airfoil.step")

## Engineering Application: CFD Domain Generation

Create a CFD domain with an airfoil and far-field boundaries.

In [ ]:
# 5. Engineering Application: CFD Domain
print("\n=== CFD Domain Generation ===")

# Create a 2D domain (extruded to 3D)
domain_length = 10.0
domain_height = 5.0
chord = 1.0

# Create the outer domain
outer_domain = cq.Workplane("XY").rect(domain_length, domain_height)

# Create the airfoil (simplified)
airfoil = (cq.Workplane("XY")
           .moveTo(0, 0)
           .lineTo(chord, 0)
           .spline([(chord*0.5, 0.1), (chord*0.2, 0.05), (0, 0)])
           .close())

# Cut the airfoil from the domain
cfd_domain = outer_domain.cut(airfoil)

# Extrude to 3D
cfd_domain_3d = cfd_domain.extrude(1.0)

print(f"Created CFD domain")
print(f"Domain size: {domain_length} x {domain_height} x 1.0")
print(f"Airfoil chord: {chord}")
print(f"Volume: {cfd_domain_3d.val().Volume():.3f}")

# Export for meshing
cq.exporters.export(cfd_domain_3d, "cfd_domain.stl")
cq.exporters.export(cfd_domain_3d, "cfd_domain.step")
print(f"\nExported CFD domain to STL and STEP")

## Exercises

1. Create a parametric turbine blade with variable thickness.
2. Generate a complex airfoil using NACA formulas.
3. Create a CFD domain with multiple obstacles.